# Text RAG Pipeline — PDF to Answer

This notebook demonstrates a complete **Retrieval-Augmented Generation (RAG)** pipeline,
running entirely locally with [Ollama](https://ollama.com/).

**RAG in a nutshell:**

1. **Retrieve** — find the most relevant passages from your documents using vector similarity search.
2. **Generate** — feed those passages as context to an LLM and let it synthesise an answer.

Instead of hoping the model memorised your content during training, RAG *gives* it the
relevant text at query time. This keeps answers grounded and reduces hallucination.

We consolidate the three separate scripts (`01_ingest.py`, `02_embed.py`, `03_query.py`)
into one interactive notebook so you can inspect every step.

In [ ]:
%pip install pymupdf sqlite-vec requests rich --quiet

## Step 0: Configuration

We support two providers:

| Provider | `PROVIDER` value | What you need running |
|----------|------------------|-----------------------|
| **Local (Ollama)** | `local` (default) | `ollama serve` with a model pulled (e.g. `ollama pull phi3:mini`) |
| **Azure AI Foundry** | `azure` | An Azure endpoint + API key set in `.env` |

If a `.env` file exists in the parent `rag/` directory it will be loaded automatically.
Otherwise the defaults below point at a local Ollama instance.

In [ ]:
import os
import struct
import sqlite3
from pathlib import Path

import requests

# ---------------------------------------------------------------------------
# Load .env from parent rag/ directory if it exists
# ---------------------------------------------------------------------------
_env_path = Path("../rag/.env")
if _env_path.exists():
    for line in _env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip())

# ---------------------------------------------------------------------------
# Provider selection
# ---------------------------------------------------------------------------
PROVIDER = os.environ.get("PROVIDER", "local")

# Ollama defaults
OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "phi3:mini")

# Azure defaults
AZURE_API_KEY = os.environ.get("AZURE_API_KEY", "")
AZURE_CHAT_BASE_URL = os.environ.get("AZURE_CHAT_BASE_URL", "").rstrip("/")
AZURE_CHAT_DEPLOYMENT = os.environ.get("AZURE_CHAT_DEPLOYMENT", "gpt-4o")
AZURE_EMBED_BASE_URL = os.environ.get("AZURE_EMBED_BASE_URL", "").rstrip("/")
AZURE_EMBED_DEPLOYMENT = os.environ.get("AZURE_EMBED_DEPLOYMENT", "text-embedding-ada-002")

print(f"Provider : {PROVIDER}")
if PROVIDER == "azure":
    print(f"Chat model : {AZURE_CHAT_DEPLOYMENT}")
    print(f"Embed model: {AZURE_EMBED_DEPLOYMENT}")
else:
    print(f"Ollama URL : {OLLAMA_BASE_URL}")
    print(f"Model      : {OLLAMA_MODEL}")

# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def _azure_headers() -> dict:
    """Return auth headers for Azure OpenAI-compatible endpoints."""
    return {
        "Authorization": f"Bearer {AZURE_API_KEY}",
        "Content-Type": "application/json",
    }


def get_embedding(text: str) -> list[float]:
    """Get an embedding vector for *text* from the configured provider."""
    if PROVIDER == "azure":
        response = requests.post(
            f"{AZURE_EMBED_BASE_URL}/embeddings",
            headers=_azure_headers(),
            json={"model": AZURE_EMBED_DEPLOYMENT, "input": text},
        )
        response.raise_for_status()
        return response.json()["data"][0]["embedding"]
    else:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/embed",
            json={"model": OLLAMA_MODEL, "input": text},
        )
        response.raise_for_status()
        return response.json()["embeddings"][0]


def chat(messages: list[dict], temperature: float = 0, max_tokens: int = 1024) -> str:
    """Send a chat-completion request and return the assistant reply."""
    if PROVIDER == "azure":
        response = requests.post(
            f"{AZURE_CHAT_BASE_URL}/chat/completions",
            headers=_azure_headers(),
            json={
                "model": AZURE_CHAT_DEPLOYMENT,
                "messages": messages,
                "temperature": temperature,
                "max_completion_tokens": max_tokens,
            },
        )
    else:
        response = requests.post(
            f"{OLLAMA_BASE_URL}/v1/chat/completions",
            json={
                "model": OLLAMA_MODEL,
                "messages": messages,
                "temperature": temperature,
                "max_completion_tokens": max_tokens,
            },
        )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

## Step 1: Ingest — PDF to Chunks

We reuse the PDFs already sitting in `../rag/docs/`. Each PDF is opened with
[PyMuPDF](https://pymupdf.readthedocs.io/), text is extracted page-by-page, and
then split into overlapping 500-character chunks.

The chunks are stored in a local SQLite database (`rag_notebook.db`) so they
don't conflict with the database used by the standalone scripts.

In [ ]:
import fitz  # pymupdf

# --- Config ---
CHUNK_SIZE = 500       # characters per chunk
CHUNK_OVERLAP = 50     # overlap between consecutive chunks
DOCS_DIR = Path("../rag/docs")
DB_FILE = Path("rag_notebook.db")


def extract_text_from_pdf(pdf_path: Path) -> list[dict]:
    """Extract text from each page of a PDF."""
    doc = fitz.open(pdf_path)
    pages = []
    for page_num in range(len(doc)):
        text = doc[page_num].get_text()
        if text.strip():
            pages.append({"page": page_num + 1, "text": text})
    doc.close()
    return pages


def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    """Split text into overlapping chunks of fixed character size."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


# --- Find PDFs ---
pdf_files = sorted(DOCS_DIR.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in {DOCS_DIR.resolve()}.\n"
        "Place a PDF there first, e.g.:\n"
        "  curl -L -o ../rag/docs/attention.pdf https://arxiv.org/pdf/1706.03762"
    )
print(f"Found {len(pdf_files)} PDF(s): {[p.name for p in pdf_files]}")

# --- Fresh DB each run ---
if DB_FILE.exists():
    DB_FILE.unlink()

db = sqlite3.connect(str(DB_FILE))
db.execute("""
    CREATE TABLE chunks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        text TEXT NOT NULL,
        source TEXT NOT NULL,
        page INTEGER NOT NULL
    )
""")

total = 0
for pdf_path in pdf_files:
    pages = extract_text_from_pdf(pdf_path)
    print(f"  {pdf_path.name}: {len(pages)} pages")
    for page_info in pages:
        for chunk in chunk_text(page_info["text"], CHUNK_SIZE, CHUNK_OVERLAP):
            db.execute(
                "INSERT INTO chunks (text, source, page) VALUES (?, ?, ?)",
                (chunk, pdf_path.name, page_info["page"]),
            )
            total += 1

db.commit()
print(f"\nTotal chunks stored: {total}")

# --- Display sample chunks ---
rows = db.execute("SELECT id, source, page, text FROM chunks LIMIT 10").fetchall()

try:
    import pandas as pd
    df = pd.DataFrame(rows, columns=["id", "source", "page", "text"])
    df["text"] = df["text"].str[:120].str.replace("\n", " ", regex=False) + "..."
    display(df)
except ImportError:
    for row in rows:
        print(f"  [{row[0]:>3}] {row[1]} p.{row[2]:>2}  {row[3][:120]}...")

if total > 10:
    print(f"\n  ... and {total - 10} more chunks")

## Step 2: Embed — Chunks to Vectors

Each text chunk is converted into a high-dimensional vector (embedding) that
captures its semantic meaning. We store the vectors in a **sqlite-vec** virtual
table inside the same database — no external vector store required.

This step calls the embedding model once per chunk, so it may take a minute
depending on the number of chunks and your hardware.

In [ ]:
import sqlite_vec

# Re-open DB with sqlite-vec loaded
db.close()
db = sqlite3.connect(str(DB_FILE))
db.enable_load_extension(True)
sqlite_vec.load(db)
db.enable_load_extension(False)

# Read all chunks
chunks = db.execute("SELECT id, text FROM chunks ORDER BY id").fetchall()
print(f"Embedding {len(chunks)} chunks...")

# Embed the first chunk to discover the vector dimension
first_vec = get_embedding(chunks[0][1])
dim = len(first_vec)
print(f"Embedding dimension: {dim}")


def serialize_float32(vector: list[float]) -> bytes:
    """Pack a list of floats into raw bytes for sqlite-vec."""
    return struct.pack(f"{len(vector)}f", *vector)


# Create the virtual table (drop first in case of re-run)
db.execute("DROP TABLE IF EXISTS vec_chunks")
db.execute(f"""
    CREATE VIRTUAL TABLE vec_chunks USING vec0(
        id INTEGER PRIMARY KEY,
        embedding FLOAT[{dim}]
    )
""")

# Insert first embedding
db.execute(
    "INSERT INTO vec_chunks (id, embedding) VALUES (?, ?)",
    (chunks[0][0], serialize_float32(first_vec)),
)

# Embed and insert the rest
for i, (chunk_id, text) in enumerate(chunks[1:], start=2):
    vector = get_embedding(text)
    db.execute(
        "INSERT INTO vec_chunks (id, embedding) VALUES (?, ?)",
        (chunk_id, serialize_float32(vector)),
    )
    if i % 20 == 0 or i == len(chunks):
        print(f"  {i}/{len(chunks)} chunks embedded", end="\r")

db.commit()
print(f"\nDone! All {len(chunks)} chunks embedded and stored.\n")

# --- Show sample embeddings (first 8 dims) ---
print("Sample embeddings (first 8 dimensions):")
print("-" * 80)
for chunk_id, text in chunks[:5]:
    vec = get_embedding(text)
    dims_str = ", ".join(f"{v:.4f}" for v in vec[:8])
    print(f"  Chunk {chunk_id:>3}: [{dims_str}, ...]")
    print(f"             {text[:60].replace(chr(10), ' ')}...")
print()

# --- DB stats ---
vec_count = db.execute("SELECT COUNT(*) FROM vec_chunks").fetchone()[0]
chunk_count = db.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
print(f"Database stats:")
print(f"  chunks     : {chunk_count} rows (plain text + metadata)")
print(f"  vec_chunks : {vec_count} rows ({dim}-dimensional vectors)")

## Step 3: Query — Search + Answer

Now the fun part. We:

1. Embed the user's question into the same vector space.
2. Use sqlite-vec to find the **top-k** closest chunks.
3. Build a prompt that includes the retrieved context.
4. Send it to the LLM and display the answer.

In [ ]:
def search_similar(db: sqlite3.Connection, query_vec: list[float], top_k: int = 3) -> list[dict]:
    """Find the top-k most similar chunks via vector distance."""
    rows = db.execute(
        """
        SELECT v.id, v.distance, c.text, c.source, c.page
        FROM vec_chunks v
        JOIN chunks c ON c.id = v.id
        WHERE v.embedding MATCH ?
            AND v.k = ?
        ORDER BY v.distance
        """,
        (serialize_float32(query_vec), top_k),
    ).fetchall()

    return [
        {"id": r[0], "distance": r[1], "text": r[2], "source": r[3], "page": r[4]}
        for r in rows
    ]


def build_messages(question: str, context_chunks: list[dict]) -> list[dict]:
    """Build the system + user messages with retrieved context."""
    context_text = "\n\n---\n\n".join(
        f"[From {c['source']}, page {c['page']}]\n{c['text']}"
        for c in context_chunks
    )
    return [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Answer the user's question based on "
                "the provided context ONLY, not internal knowledge. If the context "
                "doesn't contain enough information, say so. Cite which page the "
                "information comes from when possible."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context_text}\n\n---\n\nQuestion: {question}",
        },
    ]

In [ ]:
# --- Run a sample query ---
question = "What is the attention mechanism?"

print(f"Question: {question}\n")

# 1. Embed the question
query_vec = get_embedding(question)
print(f"Query vector dimension: {len(query_vec)}")

# 2. Search for top-3 similar chunks
results = search_similar(db, query_vec, top_k=3)

print(f"\nRetrieved {len(results)} chunks:\n")
for i, r in enumerate(results, 1):
    print(f"--- Chunk {i} (distance: {r['distance']:.4f}, source: {r['source']}, page: {r['page']}) ---")
    print(r["text"][:200].replace("\n", " "))
    print()

# 3. Build prompt and send to LLM
messages = build_messages(question, results)
print("Sending to LLM...\n")
answer = chat(messages)

print("=" * 80)
print("ANSWER")
print("=" * 80)
print(answer)

## Try your own question

Change the `question` variable below and re-run the cell to query your documents.

In [ ]:
# Change this to any question about your ingested documents
question = "How does multi-head attention work?"

# --- Retrieve & Generate ---
query_vec = get_embedding(question)
results = search_similar(db, query_vec, top_k=3)

print(f"Question: {question}\n")
print(f"Top {len(results)} retrieved chunks:")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['source']} p.{r['page']}] (dist={r['distance']:.4f}) {r['text'][:80].replace(chr(10), ' ')}...")

messages = build_messages(question, results)
answer = chat(messages)

print(f"\n{'=' * 80}")
print("ANSWER")
print(f"{'=' * 80}")
print(answer)